# Classifying ACLED farmer protests with Gemini (Batch Mode)

Template: `anzonyquispe/myneta_llm` (`table_qa.ipynb`, `response_rounds_analysis.ipynb`):
prompt with research context + worked examples, JSON answers, a 50-case pilot with a
hand-checked comparison and cost estimate, 10 rounds, and a majority vote (>= 7 of 10).

Differences: Gemini instead of OpenAI, and the full run is submitted with **Batch Mode**
(50% of the standard price). The pilot uses the standard API for immediate answers.

Run the sections in order. Sections 3-4 cost cents; section 5 is the paid batch run.

Run on 2026-10-16 (55,160 requests, 0 errors). Its outputs are kept on the cluster in `data_output/intermediate/gemini_protests/`; `build_protest2_events.py` turns `gemini_rounds.csv` into the protest2 events. ACLED event texts are not stored in this repository, so the outputs that printed them are cleared.


## 0. Setup

In [9]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Folder with outputs/ and .env; PROTESTS_5KM_DIR overrides the original location.
BASE = Path(os.environ.get("PROTESTS_5KM_DIR", Path.home() / "Downloads" / "protests_5km"))
WORK = BASE / "gemini"
BATCH_DIR = WORK / "batch"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# The API key is read from BASE/.env (GEMINI_API_KEY=...); it is never written in this notebook
load_dotenv(BASE / ".env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PILOT_MODELS = ["gemini-3.1-flash-lite", "gemini-3.5-flash"]  # compared in the pilot
MODEL = "gemini-3.1-flash-lite"  # model for the batch run: set it after the pilot
N_ROUNDS = 10
MAJORITY = 7  # a category wins when chosen in at least 7 of the 10 rounds (as in myneta_llm)
STATES = ["Punjab", "Haryana", "Uttar Pradesh", "Bihar"]
CATEGORIES = ["farmer_led", "solidarity", "against_farmers", "not_farmer", "insufficient_information"]

# Batch prices per 1M tokens (paid tier, input/output), from ai.google.dev/gemini-api/docs/pricing
BATCH_PRICES = {"gemini-3.1-flash-lite": (0.125, 0.75), "gemini-3.5-flash-lite": (0.15, 1.25)}


## 1. Events to classify
Unique ACLED events in the four study states, from the notebook run
(`outputs/acled_events_window_keyword_flag.csv`). `farm_keyword` is the manual rule (1/0).

In [2]:
events = pd.read_csv(BASE / "outputs" / "acled_events_window_keyword_flag.csv")
events = events.drop_duplicates("event_id_cnty")
events = events[events.admin1.isin(STATES)].reset_index(drop=True)
events.to_csv(WORK / "events_to_classify.csv", index=False)
print(f"{len(events):,} events | keyword rule:", events.farm_keyword.value_counts().to_dict())

5,516 events | keyword rule: {1: 2909, 0: 2607}


## 2. Prompt
Instructions + worked examples (made-up events, not taken from the data) + a JSON schema
that every answer must follow. **Review the category definitions with the team before the pilot.**

In [3]:
SYSTEM_INSTRUCTION = """You are a research assistant studying the farmer protests in North India between \
June 2020 and December 2021 (the movement against the three farm laws). You will receive one protest \
event from the ACLED database. Classify it into exactly one category:
- farmer_led: farmers, farm workers or farmer organizations (e.g., kisan unions such as Bharatiya Kisan \
Union or Samyukt Kisan Morcha) take part in the protest, including a counter-protest by farmers reported \
in the same event.
- solidarity: the protesters are other groups (political parties, students, trade unions, lawyers, etc.) \
and the protest supports farmers or their demands, but farmers are not reported as taking part.
- against_farmers: the protest opposes the farmers' movement or supports the farm laws.
- not_farmer: the protest is not about farmers or agriculture (the word "farm" alone, e.g. a farmhouse, \
is not enough).
- insufficient_information: the text does not allow a decision.
Use only the information in the event. Return a JSON object with event_id, category and a \
one-sentence reasoning."""

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "event_id": {"type": "STRING"},
        "category": {"type": "STRING", "enum": CATEGORIES},
        "reasoning": {"type": "STRING"},
    },
    "required": ["event_id", "category", "reasoning"],
    "propertyOrdering": ["event_id", "category", "reasoning"],
}

# Worked examples (illustrative, invented for the prompt)
FEW_SHOT = [
    ("event_id: EX1\ndate: 2020-11-27\nlocation: Ambala, Ambala, Haryana\nactor: Protesters (India)\n"
     "associated actor: bku: bharatiya kisan union; farmers (india)\n"
     "notes: On 27 November 2020, BKU members and farmers held a tractor march in Ambala against the farm laws.",
     {"event_id": "EX1", "category": "farmer_led",
      "reasoning": "Farmers and a kisan union take part in a protest against the farm laws."}),
    ("event_id: EX2\ndate: 2020-12-05\nlocation: Patna, Patna, Bihar\nactor: Protesters (India)\n"
     "associated actor: nsui: national students union of india\n"
     "notes: On 5 December 2020, NSUI students protested in Patna in support of the farmers agitating against the farm laws.",
     {"event_id": "EX2", "category": "solidarity",
      "reasoning": "Students protest in support of farmers; farmers are not reported as taking part."}),
    ("event_id: EX3\ndate: 2021-01-10\nlocation: Karnal, Karnal, Haryana\nactor: Protesters (India)\n"
     "associated actor: bjp: bharatiya janata party\n"
     "notes: On 10 January 2021, BJP workers marched in Karnal in support of the farm laws and against the farmers' highway blockade.",
     {"event_id": "EX3", "category": "against_farmers",
      "reasoning": "The protest supports the farm laws and opposes the farmers' blockade."}),
    ("event_id: EX4\ndate: 2021-03-03\nlocation: Mohali, Sahibzada Ajit Singh Nagar, Punjab\nactor: Protesters (India)\n"
     "associated actor: teachers (india)\n"
     "notes: On 3 March 2021, contract teachers protested outside a minister's farmhouse in Mohali demanding regular jobs.",
     {"event_id": "EX4", "category": "not_farmer",
      "reasoning": "A teachers' job protest; the farmhouse is only the location."}),
]


def event_text(row):
    """Text sent to the model for one ACLED event."""
    return (f"event_id: {row.event_id_cnty}\ndate: {row.event_date}\n"
            f"location: {row.location}, {row.admin2}, {row.admin1}\n"
            f"actor: {row.actor1}\nassociated actor: {row.assoc_actor_1}\nnotes: {row.notes}")


def build_contents(row):
    """Worked examples as user/model turns, then the event (same idea as FEW_SHOT_PROMPT in myneta_llm)."""
    contents = []
    for text, answer in FEW_SHOT:
        contents.append({"role": "user", "parts": [{"text": text}]})
        contents.append({"role": "model", "parts": [{"text": json.dumps(answer)}]})
    contents.append({"role": "user", "parts": [{"text": event_text(row)}]})
    return contents


def generation_config(model):
    """Generation settings shared by the pilot and the batch requests."""
    cfg = {"response_mime_type": "application/json", "response_schema": RESPONSE_SCHEMA}
    if model.startswith("gemini-2.5-flash"):
        cfg["thinking_config"] = {"thinking_budget": 0}  # no billed "thinking" for a simple label
    return cfg


print(event_text(events.iloc[0]))

## 3. Single-event test (standard API, ~cost of a fraction of a cent)

In [6]:
def classify(row, model):
    """One standard (non-batch) call; returns the parsed answer and token usage."""
    cfg = generation_config(model)
    response = client.models.generate_content(
        model=model,
        contents=build_contents(row),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type=cfg["response_mime_type"],
            response_schema=types.Schema.model_validate(cfg["response_schema"]),
            thinking_config=(types.ThinkingConfig(**cfg["thinking_config"]) if "thinking_config" in cfg else None),
        ),
    )
    usage = response.usage_metadata
    return json.loads(response.text), {
        "input_tokens": usage.prompt_token_count,
        "output_tokens": (usage.candidates_token_count or 0) + (usage.thoughts_token_count or 0),
    }


test_row = events.iloc[0]
answer, usage = classify(test_row, PILOT_MODELS[0])
print(event_text(test_row), "\n")
print(answer, usage)

## 4. Pilot: 50 events (25 selected by the keyword rule, 25 not)
1. Run the next cell: it saves `pilot_to_label.csv`.
2. **Open it in Excel, fill `my_category` yourself (without looking at Gemini's answers) and save it as `pilot_labeled.csv`.**
3. Run the cell after: both pilot models classify the same 50 events and the cost of the full batch run is projected.
4. Run the accuracy cell.

In [7]:
pilot = pd.concat([
    events[events.farm_keyword == 1].sample(25, random_state=42),
    events[events.farm_keyword == 0].sample(25, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)
to_label = pilot[["event_id_cnty", "event_date", "admin1", "location", "actor1", "assoc_actor_1", "notes"]].copy()
to_label["my_category"] = ""  # fill with one of CATEGORIES
to_label.to_csv(WORK / "pilot_to_label.csv", index=False)
print("Saved", WORK / "pilot_to_label.csv", "| categories:", CATEGORIES)

In [8]:
rows = []
for model in PILOT_MODELS:
    for _, row in pilot.iterrows():
        try:
            answer, usage = classify(row, model)
            rows.append({"model": model, "event_id_cnty": row.event_id_cnty, **answer, **usage})
        except Exception as error:  # keep going; failed calls are listed below
            rows.append({"model": model, "event_id_cnty": row.event_id_cnty, "error": str(error)})
        time.sleep(0.5)
pilot_results = pd.DataFrame(rows)
pilot_results.to_csv(WORK / "pilot_gemini.csv", index=False)
print("failed calls:", int(pilot_results.get("error", pd.Series(dtype=str)).notna().sum()))

# Project the cost of the full batch run from the real token counts
for model, g in pilot_results.dropna(subset=["input_tokens"]).groupby("model"):
    price_in, price_out = BATCH_PRICES[model]
    n_requests = len(events) * N_ROUNDS
    cost = n_requests * (g.input_tokens.mean() * price_in + g.output_tokens.mean() * price_out) / 1e6
    print(f"{model}: {g.input_tokens.mean():.0f} in / {g.output_tokens.mean():.0f} out tokens per call "
          f"-> batch run of {n_requests:,} requests ~ ${cost:.2f}")

failed calls: 0
gemini-3.1-flash-lite: 836 in / 55 out tokens per call -> batch run of 55,160 requests ~ $10.73
gemini-3.5-flash: 836 in / 323 out tokens per call -> batch run of 55,160 requests ~ $50.67


In [10]:
# Accuracy against your hand labels (run after saving pilot_labeled.csv)
labels = pd.read_csv(WORK / "pilot_labeled.csv")[["event_id_cnty", "my_category"]]
check = pd.read_csv(WORK / "pilot_gemini.csv").merge(labels, on="event_id_cnty")
check["match"] = check.category == check.my_category
print(check.groupby("model").match.mean().rename("accuracy"))
print(pd.crosstab([check.model, check.my_category], check.category))

model
gemini-3.1-flash-lite    1.00
gemini-3.5-flash         0.98
Name: accuracy, dtype: float64
category                               against_farmers  farmer_led  \
model                 my_category                                    
gemini-3.1-flash-lite against_farmers                1           0   
                      farmer_led                     0          22   
                      not_farmer                     0           0   
                      solidarity                     0           0   
gemini-3.5-flash      against_farmers                0           0   
                      farmer_led                     0          22   
                      not_farmer                     0           0   
                      solidarity                     0           0   

category                               not_farmer  solidarity  
model                 my_category                              
gemini-3.1-flash-lite against_farmers           0           0  
          

## 5. Full run with Batch Mode (the discounted, paid step)
One JSONL file and one batch job per round (10 jobs of ~5,500 requests). Each request has
`key = <event_id>_r<round>`. Job names are saved in `batch/batch_jobs.csv` **as soon as they are
created**, and rounds already submitted are skipped, so re-running the cell never pays twice.

Run 5a (build files) and 5b (tiny 5-request batch to check the format) before 5c.

In [11]:
# 5a. Build one JSONL request file per round
def batch_request(row, round_no, model):
    return {
        "key": f"{row.event_id_cnty}_r{round_no:02d}",
        "request": {
            "contents": build_contents(row),
            "system_instruction": {"parts": [{"text": SYSTEM_INSTRUCTION}]},
            "generation_config": generation_config(model),
        },
    }


for round_no in range(1, N_ROUNDS + 1):
    path = BATCH_DIR / f"requests_round_{round_no:02d}.jsonl"
    with open(path, "w") as f:
        for _, row in events.iterrows():
            f.write(json.dumps(batch_request(row, round_no, MODEL)) + "\n")
print("JSONL files:", sorted(p.name for p in BATCH_DIR.glob("requests_round_*.jsonl")))

JSONL files: ['requests_round_01.jsonl', 'requests_round_02.jsonl', 'requests_round_03.jsonl', 'requests_round_04.jsonl', 'requests_round_05.jsonl', 'requests_round_06.jsonl', 'requests_round_07.jsonl', 'requests_round_08.jsonl', 'requests_round_09.jsonl', 'requests_round_10.jsonl']


In [12]:
# Helpers: submit a JSONL file as a batch job, wait for it, and parse its results
def submit_batch(path, display_name):
    uploaded = client.files.upload(file=str(path),
                                   config=types.UploadFileConfig(display_name=display_name, mime_type="jsonl"))
    job = client.batches.create(model=MODEL, src=uploaded.name, config={"display_name": display_name})
    return job.name


DONE = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}


def wait_for(job_name, every=60):
    while True:
        job = client.batches.get(name=job_name)
        if job.state.name in DONE:
            return job
        print(time.strftime("%H:%M:%S"), job_name, job.state.name)
        time.sleep(every)


def parse_results(job):
    """One row per request: key, event, round, category, reasoning, tokens or error."""
    content = client.files.download(file=job.dest.file_name).decode("utf-8")
    rows = []
    for line in content.splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        event_id, round_tag = item["key"].rsplit("_r", 1)
        row = {"key": item["key"], "event_id_cnty": event_id, "round": int(round_tag)}
        try:
            response = item["response"]
            answer = json.loads(response["candidates"][0]["content"]["parts"][0]["text"])
            usage = response.get("usageMetadata", {})
            row.update(category=answer.get("category"), reasoning=answer.get("reasoning"),
                       input_tokens=usage.get("promptTokenCount"),
                       output_tokens=usage.get("candidatesTokenCount", 0) + usage.get("thoughtsTokenCount", 0))
        except (KeyError, IndexError, json.JSONDecodeError, TypeError):
            row["error"] = json.dumps(item.get("error", item.get("response")))[:500]
        rows.append(row)
    return pd.DataFrame(rows)

In [13]:
# 5b. Format check: a 5-request batch (costs a fraction of a cent; may take minutes to hours)
smoke_path = BATCH_DIR / "smoke_test.jsonl"
with open(smoke_path, "w") as f:
    for _, row in events.head(5).iterrows():
        f.write(json.dumps(batch_request(row, 0, MODEL)) + "\n")
smoke_job = wait_for(submit_batch(smoke_path, "protests-smoke-test"))
print(smoke_job.state.name)
if smoke_job.state.name == "JOB_STATE_SUCCEEDED":
    display(parse_results(smoke_job))

In [14]:
# 5c. Submit the 10 rounds (skips rounds already submitted)
jobs_file = BATCH_DIR / "batch_jobs.csv"
jobs = pd.read_csv(jobs_file) if jobs_file.exists() else pd.DataFrame(columns=["round", "job_name", "model"])
for round_no in range(1, N_ROUNDS + 1):
    if round_no in set(jobs["round"].astype(int)):
        print(f"round {round_no:02d}: already submitted, skipped")
        continue
    name = submit_batch(BATCH_DIR / f"requests_round_{round_no:02d}.jsonl", f"protests-round-{round_no:02d}")
    jobs = pd.concat([jobs, pd.DataFrame([{"round": round_no, "job_name": name, "model": MODEL}])])
    jobs.to_csv(jobs_file, index=False)  # saved right away, before the next submission
    print(f"round {round_no:02d}: submitted {name}")

round 01: submitted batches/fv48fdlll8ui1lcuxqvrfzkar5h05bf5ltoa
round 02: submitted batches/8e5ae1u19fasqgkleonr5jxg7ni4u632fehh
round 03: submitted batches/pf9dqy55nv72spfuim3boa2z8q0ure0nxfsw
round 04: submitted batches/9sauytoua6ktwi3mjhvtfp5vymreoxfx31bt
round 05: submitted batches/0ni6lhbmvwbfj56ba9yahn5hmgeulwi3tx5j
round 06: submitted batches/i1m2o0x0uqqwxp53karsau685ovr3tb7fid1
round 07: submitted batches/k5qekqhro5mj3k49wkglt5l5clcxys1ozh5m
round 08: submitted batches/qwbaaesps5lwdywpdmlvcn0lbogdkg37vo02
round 09: submitted batches/0qe998see6ond80zyf2gj34a3imjyfkwoc99
round 10: submitted batches/zin855hon4l0i2dhhuiotkxypuy47ztkoa0f


In [16]:
# 5d. Check status (re-run until all rounds are JOB_STATE_SUCCEEDED)
jobs = pd.read_csv(BATCH_DIR / "batch_jobs.csv")
for _, j in jobs.iterrows():
    print(f"round {int(j['round']):02d}", client.batches.get(name=j.job_name).state.name)

round 01 JOB_STATE_SUCCEEDED
round 02 JOB_STATE_SUCCEEDED
round 03 JOB_STATE_SUCCEEDED
round 04 JOB_STATE_SUCCEEDED
round 05 JOB_STATE_SUCCEEDED
round 06 JOB_STATE_SUCCEEDED
round 07 JOB_STATE_SUCCEEDED
round 08 JOB_STATE_SUCCEEDED
round 09 JOB_STATE_SUCCEEDED
round 10 JOB_STATE_SUCCEEDED


In [17]:
# 5e. Download and parse all rounds
jobs = pd.read_csv(BATCH_DIR / "batch_jobs.csv")
parts = []
for _, j in jobs.iterrows():
    job = client.batches.get(name=j.job_name)
    if job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"round {int(j['round']):02d} not ready: {job.state.name}")
        continue
    result = parse_results(job)
    result.to_csv(BATCH_DIR / f"results_round_{int(j['round']):02d}.csv", index=False)
    parts.append(result)
rounds = pd.concat(parts, ignore_index=True)
rounds["model"] = MODEL
rounds.to_csv(WORK / "gemini_rounds.csv", index=False)
print(f"{len(rounds):,} answers | errors: {int(rounds.get('error', pd.Series(dtype=str)).notna().sum())}")

55,160 answers | errors: 0


## 6. Majority vote (>= 7 of 10 rounds, as in myneta_llm)

In [18]:
rounds = pd.read_csv(WORK / "gemini_rounds.csv").dropna(subset=["category"])
votes = rounds.groupby(["event_id_cnty", "category"]).size().rename("votes").reset_index()
top = votes.sort_values("votes", ascending=False).drop_duplicates("event_id_cnty")
top["gemini_category"] = top.category.where(top.votes >= MAJORITY, "no_majority")
majority = top[["event_id_cnty", "gemini_category", "votes"]]
majority.to_csv(WORK / "gemini_majority.csv", index=False)
print(majority.gemini_category.value_counts())

gemini_category
not_farmer                  2601
farmer_led                  2538
solidarity                   312
against_farmers               38
no_majority                   19
insufficient_information       8
Name: count, dtype: int64


## 7. Comparison with the manual keyword rule (step 3 of the plan)
Which Gemini categories count as a "farmer protest" is a team decision; the crosstab shows all of them.

In [19]:
comparison = events[["event_id_cnty", "farm_keyword", "admin1", "assoc_actor_1", "notes"]].merge(
    pd.read_csv(WORK / "gemini_majority.csv"), on="event_id_cnty", how="left")
comparison["gemini_category"] = comparison.gemini_category.fillna("no_answer")  # every round failed
print(pd.crosstab(comparison.gemini_category, comparison.farm_keyword, margins=True))
comparison.to_csv(WORK / "comparison_keyword_vs_gemini.csv", index=False)

# Disagreements to read by hand
disagree = comparison[((comparison.farm_keyword == 1) & ~comparison.gemini_category.isin(["farmer_led"]))
                      | ((comparison.farm_keyword == 0) & comparison.gemini_category.isin(["farmer_led", "solidarity"]))]
disagree.to_csv(WORK / "disagreements.csv", index=False)
print(f"{len(disagree):,} disagreements saved to disagreements.csv")

farm_keyword                 0     1   All
gemini_category                           
against_farmers              1    37    38
farmer_led                   2  2536  2538
insufficient_information     6     2     8
no_majority                  9    10    19
not_farmer                2567    34  2601
solidarity                  22   290   312
All                       2607  2909  5516
397 disagreements saved to disagreements.csv
